# Titanic Dataset — EDA with YData Profiling

**Author**: Prajwal Nerkar  
**Dataset**: [Titanic-Dataset.csv](https://github.com/HarshvardhanSingh-13/Datasets/tree/main/Titanic_Dataset)  
**Tool**: [YData Profiling](https://github.com/ydataai/ydata-profiling)

---

## Workflow

| Step | Description |
|------|-------------|
| 1 | Setup & Imports |
| 2 | Load Environment Variables |
| 3 | Download Dataset from GitHub |
| 4 | Data Overview |
| 5 | Missing Values Analysis |
| 6 | YData Profiling — Full EDA Report |
| 7 | Export Report |


---
## Step 1 — Setup & Imports

In [ ]:
# Install dependencies if running for the first time
# Uncomment the line below if packages are not installed
# !pip install -r requirements.txt

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import StringIO
from dotenv import load_dotenv
from ydata_profiling import ProfileReport
from IPython.display import display, HTML

print('All imports successful!')
print(f'Pandas version  : {pd.__version__}')
print(f'NumPy version   : {np.__version__}')

---
## Step 2 — Load Environment Variables

Configuration is stored in `.env` (git-ignored).  
Copy `.env.example` → `.env` and fill in your values before running.

In [ ]:
load_dotenv()

GITHUB_USERNAME  = os.getenv('GITHUB_USERNAME', 'not_set')
GITHUB_TOKEN     = os.getenv('GITHUB_TOKEN', '')
DATASET_URL      = os.getenv(
    'DATASET_URL',
    'https://raw.githubusercontent.com/HarshvardhanSingh-13/Datasets/main/Titanic_Dataset/Titanic-Dataset.csv'
)
PROJECT_NAME     = os.getenv('PROJECT_NAME', 'Titanic EDA - YData Profiling')
PROJECT_AUTHOR   = os.getenv('PROJECT_AUTHOR', 'Prajwal Nerkar')

print(f'Project  : {PROJECT_NAME}')
print(f'Author   : {PROJECT_AUTHOR}')
print(f'GitHub   : {GITHUB_USERNAME}')
print(f'Dataset  : {DATASET_URL}')
print(f'Token    : {"set" if GITHUB_TOKEN and GITHUB_TOKEN != "your_personal_access_token_here" else "NOT set (ok for public repos)"}')

---
## Step 3 — Download Dataset from GitHub

In [ ]:
headers = {}
if GITHUB_TOKEN and GITHUB_TOKEN != 'your_personal_access_token_here':
    headers['Authorization'] = f'token {GITHUB_TOKEN}'

response = requests.get(DATASET_URL, headers=headers)
response.raise_for_status()

df = pd.read_csv(StringIO(response.text))

print(f'Dataset loaded successfully!')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

---
## Step 4 — Data Overview

In [ ]:
print('=== First 5 rows ===')
display(df.head())

In [ ]:
print('=== Column Data Types ===')
display(df.dtypes.to_frame('dtype').T)

print('\n=== Basic Statistics ===')
display(df.describe(include='all'))

In [ ]:
print('=== Dataset Info ===')
df.info()

---
## Step 5 — Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print('=== Columns with Missing Values ===')
display(missing_df)

# Heatmap
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis', ax=ax)
ax.set_title('Missing Values Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Survival distribution — quick sanity check
if 'Survived' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    df['Survived'].value_counts().plot.pie(
        labels=['Did not survive', 'Survived'],
        autopct='%1.1f%%',
        colors=['#e74c3c', '#2ecc71'],
        ax=axes[0]
    )
    axes[0].set_title('Survival Distribution')
    axes[0].set_ylabel('')

    sns.countplot(data=df, x='Pclass', hue='Survived',
                  palette={0: '#e74c3c', 1: '#2ecc71'}, ax=axes[1])
    axes[1].set_title('Survival by Passenger Class')
    axes[1].legend(title='Survived', labels=['No', 'Yes'])

    plt.tight_layout()
    plt.show()

---
## Step 6 — YData Profiling — Full Automated EDA Report

> This generates a comprehensive HTML report covering distributions, correlations, missing values, interactions, and more.

In [ ]:
profile = ProfileReport(
    df,
    title=f'{PROJECT_NAME} — by {PROJECT_AUTHOR}',
    explorative=True,
    dark_mode=False,
    correlations={
        'auto': {'calculate': True},
        'pearson': {'calculate': True},
        'spearman': {'calculate': True},
        'kendall': {'calculate': True},
        'phi_k': {'calculate': True},
    },
    missing_diagrams={
        'bar': True,
        'matrix': True,
        'heatmap': True,
    },
    interactions={
        'continuous': True,
    }
)

print('Profile report generated!')

In [ ]:
# Render the report inline in the notebook
profile.to_notebook_iframe()

---
## Step 7 — Export Report as HTML

In [ ]:
report_path = 'titanic_profile_report.html'
profile.to_file(report_path)

print(f'Report saved to: {report_path}')
display(HTML(f'<a href="{report_path}" target="_blank">Open full report in new tab</a>'))

---

## Summary

| Item | Detail |
|------|--------|
| Dataset | Titanic passenger data (~891 rows) |
| Source | [HarshvardhanSingh-13/Datasets](https://github.com/HarshvardhanSingh-13/Datasets) |
| Tool | YData Profiling |
| Output | `titanic_profile_report.html` |
| Author | Prajwal Nerkar |

> The HTML report is git-ignored. Share it separately or host it via GitHub Pages.